# Laboratorio 7 — Spark MLlib
**CC3066 Data Science — Universidad del Valle de Guatemala — Semestre II 2026**

Análisis de salarios de personas asalariadas con la **Encuesta Nacional de Empleo e Ingresos Continua (ENEIC)** del INE.

- **Desarrollo / entrenamiento:** trimestres I–IV de 2025.
- **Prueba final:** trimestre I de 2026.

Todo el procesamiento analítico se realiza con **PySpark 3.5 (API de DataFrames / `pyspark.ml`)**. Los archivos Excel se leen con `openpyxl` (Spark no tiene lector nativo de Excel), se convierten a DataFrames de Spark con esquema explícito y se guardan en Parquet. A pandas sólo se transfieren tablas agregadas o muestras (≤ 10,000 registros) para graficar.

**Contenido:** secciones 1–4 (análisis exploratorio y segmentación) y 5–8 (modelado supervisado: regresión lineal, Random Forest, evaluación en 2026 y análisis de errores).

**Ejecución:** el notebook corre de principio a fin sin pasos manuales. Si los Excel no están en `data/raw/`, se descargan automáticamente del sitio del INE. Ver `README.md` para levantar el ambiente con Docker.

```
data/raw/        <- archivos .xlsx del INE (se descargan si faltan; no versionados)
data/parquet/    <- salidas Parquet generadas por este notebook
modelos/         <- pipelines de MLlib guardados
```

In [1]:
import os, json, math, glob, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import openpyxl

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.stat import Correlation
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator, RegressionEvaluator
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.regression import LinearRegression, RandomForestRegressor

%matplotlib inline
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
SEED = 42

spark = (SparkSession.builder
         .appName("Lab7-ENEIC")
         .master("local[*]")
         .config("spark.driver.memory", "6g")
         .config("spark.sql.shuffle.partitions", "16")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 23:47:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.1


---
# 1. Carga, armonización y calidad de datos

## 1.1 Archivos y columnas requeridas

Cada archivo se identifica por su **procedencia** (no por la columna `TRIMESTRE`). Se crean `archivo_origen`, `periodo_archivo`, `anio_archivo` y `trimestre_calendario` a partir de esta tabla.

In [2]:
RAW_DIR = "data/raw"
PQ_DIR = "data/parquet"
os.makedirs(PQ_DIR, exist_ok=True)

ARCHIVOS = [
    # archivo, periodo_archivo, anio_archivo, trimestre_calendario, uso
    ("Personas_ENEIC_T1_2025.xlsx",                 "2025T1", 2025, 1, "train"),
    ("Personas-ENEIC-T2-2025.xlsx",                 "2025T2", 2025, 2, "train"),
    ("Base-de-datos-Personas-ENEIC-III-2025.xlsx",  "2025T3", 2025, 3, "train"),
    ("Base-de-datos-Personas-ENEIC-IV-2025.xlsx",   "2025T4", 2025, 4, "validación"),
    ("Base-de-datos-Personas-ENEIC-I-2026.xlsx",    "2026T1", 2026, 1, "test"),
]

# Fuente oficial: https://www.ine.gob.gt/encuesta-nacional-de-empleo-e-ingresos/
URLS_INE = {
    "Personas_ENEIC_T1_2025.xlsx": "https://www.ine.gob.gt/wp-content/uploads/2026/01/Personas_ENEIC_T1_2025.xlsx",
    "Personas-ENEIC-T2-2025.xlsx": "https://www.ine.gob.gt/wp-content/uploads/2026/01/Personas-ENEIC-T2-2025.xlsx",
    "Base-de-datos-Personas-ENEIC-III-2025.xlsx": "https://www.ine.gob.gt/wp-content/uploads/2026/05/Base-de-datos-Personas-ENEIC-III-2025.xlsx",
    "Base-de-datos-Personas-ENEIC-IV-2025.xlsx": "https://www.ine.gob.gt/wp-content/uploads/2026/06/Base-de-datos-Personas-ENEIC-IV-2025.xlsx",
    "Base-de-datos-Personas-ENEIC-I-2026.xlsx": "https://www.ine.gob.gt/wp-content/uploads/2026/09/Base-de-datos-Personas-ENEIC-I-2026.xlsx",
    "Diccionario-Personas-ENEIC-I-2026.xlsx": "https://www.ine.gob.gt/wp-content/uploads/2026/09/Diccionario-Personas-ENEIC-I-2026.xlsx",
}

import urllib.request
os.makedirs(RAW_DIR, exist_ok=True)
for nombre, url in URLS_INE.items():
    ruta = f"{RAW_DIR}/{nombre}"
    if not os.path.exists(ruta):
        print("Descargando", nombre, "...")
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(ruta + ".part", "wb") as f:
            f.write(r.read())
        os.replace(ruta + ".part", ruta)
print("Archivos en data/raw:", sorted(os.listdir(RAW_DIR)))

# Columnas originales requeridas
COLS_ORIG = ["ANIO", "TRIMESTRE", "DOMINIO", "NUM_HOGAR", "NUM_PERSONA", "FACTOR",
             "OCUPADOS", "P02A03", "P05C07A", "P05C07B", "P05H01A",
             "P03A03A", "P05C16", "P05D01"]

Archivos en data/raw: ['Base-de-datos-Personas-ENEIC-I-2026.xlsx', 'Base-de-datos-Personas-ENEIC-III-2025.xlsx', 'Base-de-datos-Personas-ENEIC-IV-2025.xlsx', 'Diccionario-Personas-ENEIC-I-2026.xlsx', 'Personas-ENEIC-T2-2025.xlsx', 'Personas_ENEIC_T1_2025.xlsx']


## 1.2 Lectura individual de cada Excel → Parquet crudo

Cada archivo se procesa **por separado** para controlar memoria:

1. `openpyxl` en modo `read_only` recorre las filas y extrae **sólo** las 14 columnas requeridas, localizándolas **por nombre** en el encabezado (no por posición).
2. Como un mismo código puede llegar como número (`2`, `2.0`) o como texto (`'2'`, `' 2 '`), todo valor se normaliza a una **cadena canónica** (`2.0 → "2"`, espacios recortados, vacío → nulo). Así el DataFrame de Spark se crea con un esquema explícito homogéneo (`StringType`) y la conversión numérica se hace después, de forma idéntica para los cinco archivos.
3. El resultado se guarda en `data/parquet/crudo/<periodo>` (si ya existe, se reutiliza para no releer el Excel).

In [3]:
def normalizar(v):
    # Representación canónica en texto de un valor de celda
    if v is None:
        return None
    if isinstance(v, bool):
        return str(int(v))
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
        return repr(v)          # conserva 'nan' / 'inf' para poder detectarlos después
    s = str(v).strip()
    return s if s != "" else None

ESQUEMA_CRUDO = T.StructType([T.StructField(c, T.StringType(), True) for c in COLS_ORIG])

def excel_a_parquet(archivo, periodo):
    destino = f"{PQ_DIR}/crudo/{periodo}"
    meta_path = f"{PQ_DIR}/crudo/{periodo}_meta.json"
    if os.path.exists(destino) and os.path.exists(meta_path):
        with open(meta_path) as f:
            return json.load(f)
    t0 = time.time()
    wb = openpyxl.load_workbook(f"{RAW_DIR}/{archivo}", read_only=True, data_only=True)
    ws = wb[wb.sheetnames[0]]
    filas = ws.iter_rows(values_only=True)
    encabezado = [str(h).strip().upper() if h is not None else "" for h in next(filas)]
    idx = {c: encabezado.index(c) for c in COLS_ORIG}
    datos = []
    for fila in filas:
        if fila is None or all(v is None for v in fila):
            continue            # filas completamente vacías al final de la hoja
        datos.append(tuple(normalizar(fila[idx[c]]) for c in COLS_ORIG))
    wb.close()
    sdf = spark.createDataFrame(datos, schema=ESQUEMA_CRUDO)
    sdf.write.mode("overwrite").parquet(destino)
    meta = {"archivo": archivo, "hoja": ws.title, "n_columnas": len([h for h in encabezado if h]),
            "n_filas": len(datos), "posiciones": {c: idx[c] + 1 for c in COLS_ORIG}}
    with open(meta_path, "w") as f:
        json.dump(meta, f)
    print(f"{periodo}: {len(datos):,} filas leídas en {time.time()-t0:.0f}s")
    return meta

META = {p: excel_a_parquet(a, p) for a, p, *_ in ARCHIVOS}
pd.DataFrame({p: {"hoja": m["hoja"], "columnas originales": m["n_columnas"], "registros originales": m["n_filas"]}
              for p, m in META.items()}).T

,hoja,columnas originales,registros originales
2025T1,Personas_ENEIC_T1_2025,270,51588
2025T2,Personas ENEIC T2-2025,270,51167
2025T3,Personas ENEIC T3-2025,270,51583
2025T4,Base de datos Personas ENEIC IV,302,49338
2026T1,Personas_ENEIC_T1-2026,270,49843


Los conteos coinciden con los de la tabla del enunciado (51,588 / 51,167 / 51,583 / 49,338 / 49,843) y el archivo IV‑2025 tiene 302 columnas frente a 270 de los demás.

### ¿Por qué IV‑2025 no puede apilarse por posición?
La siguiente tabla muestra la **posición (1‑based)** de cada columna requerida dentro de su archivo:

In [4]:
posiciones = pd.DataFrame({p: m["posiciones"] for p, m in META.items()})
posiciones["¿misma posición en todos?"] = posiciones.nunique(axis=1).eq(1)
posiciones

,2025T1,2025T2,2025T3,2025T4,2026T1,¿misma posición en todos?
ANIO,1,1,1,1,1,True
TRIMESTRE,2,2,2,2,2,True
DOMINIO,3,3,3,3,3,True
NUM_HOGAR,4,4,4,4,4,True
NUM_PERSONA,7,7,7,7,7,True
FACTOR,5,5,5,5,5,True
OCUPADOS,266,266,266,298,266,False
P02A03,13,13,13,9,13,False
P05C07A,68,68,68,61,68,False
P05C07B,69,69,69,62,69,False


IV‑2025 tiene **32 columnas adicionales** y un orden distinto: por ejemplo la edad (`P02A03`) está en la columna 9 en lugar de la 13, y `OCUPADOS` en la 298 en lugar de la 266. Un `union()` posicional pegaría la edad de un archivo con otra variable del otro (p. ej. un código de residencia), produciendo datos silenciosamente corruptos (o fallaría por diferente número de columnas). Por eso se selecciona cada variable **por nombre** y se une con **`unionByName`**, que empareja columnas por su nombre y no por su posición.